# 5주차 예제 — Loan Prediction 분류 (Classification)

1~4주차에는 숫자를 예측하는 **회귀(regression)** 를 다뤘습니다. 이번 주에는 범주를 예측하는 **분류(classification)** 로 넘어가 대출 신청자의 정보를 바탕으로 승인(Y)과 거절(N)을 예측합니다.

1. 분류 평가지표란 무엇인가 (간단한 예제로 원리부터)
2. 데이터 불러오기 & 타겟 분포 확인
3. 결측치 확인과 전처리 대상 지정
4. 원본 입력과 타겟 준비
5. 학습/검증 데이터 분리
6. Logistic Regression 학습
7. 예측 결과와 Confusion Matrix
8. accuracy / precision / recall / F1
9. 불균형 데이터 확인 — 다수 클래스 기준선과 비교


## Part 1. 분류 평가지표란 무엇인가 (간단한 예제)

10명의 합격 여부를 예측했다고 가정합니다. 실제 합격 6명, 불합격 4명인데, 모델이 이렇게 예측했습니다.


In [ ]:
import pandas as pd

toy = pd.DataFrame({
    '실제': [1, 1, 1, 1, 1, 1, 0, 0, 0, 0],  # 1=합격, 0=불합격
    '예측': [1, 1, 1, 1, 0, 0, 0, 0, 0, 1],
})
toy


### Confusion Matrix — 네 칸으로 구분합니다

예측과 실제가 일치하는지, 일치하지 않는다면 어떤 방향의 차이인지에 따라 네 가지 경우로 나눕니다.

| | 예측: 합격(1) | 예측: 불합격(0) |
|---|---|---|
| **실제: 합격(1)** | TP (맞게 합격 예측) | FN (합격인데 불합격으로 예측 — 놓침) |
| **실제: 불합격(0)** | FP (불합격인데 합격으로 예측 — 오탐) | TN (맞게 불합격 예측) |


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(toy['실제'], toy['예측'])
print(cm)
# cm[0][0]=TN, cm[0][1]=FP, cm[1][0]=FN, cm[1][1]=TP 순서입니다


In [ ]:
tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}, FP={fp}, FN={fn}, TN={tn}')


### 네 칸으로 지표를 직접 계산합니다

- **accuracy** = (TP+TN) / 전체 — 전체 중 맞춘 비율
- **precision** = TP / (TP+FP) — '합격'이라 예측한 것 중 실제 합격 비율
- **recall** = TP / (TP+FN) — 실제 합격자 중 모델이 찾아낸 비율
- **F1** = 2 × (precision × recall) / (precision + recall) — 둘의 조화평균


In [ ]:
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print(f'accuracy={accuracy:.3f}, precision={precision:.3f}, recall={recall:.3f}, f1={f1:.3f}')


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print('sklearn 값:')
print(accuracy_score(toy['실제'], toy['예측']))
print(precision_score(toy['실제'], toy['예측']))
print(recall_score(toy['실제'], toy['예측']))
print(f1_score(toy['실제'], toy['예측']))


직접 계산한 값과 sklearn 값이 정확히 일치합니다. **accuracy는 전체 정답 비율을 하나의 숫자로 요약하고, precision과 recall은 서로 다른 오류를 각각 보여줍니다.** 이 간단한 예제에서 recall(0.667)이 precision(0.8)보다 낮은 이유는 실제 합격자 6명 중 2명을 '불합격'으로 예측한 FN이 있기 때문입니다. 오류의 영향은 상황에 따라 달라집니다. 합격자를 놓치는 FN과 불합격자를 합격으로 예측하는 FP가 각각 어떤 비용을 만드는지 생각하면 우선할 지표를 선택하기 쉬워집니다.


## Part 2. 데이터 불러오기 & 타겟 분포 확인

614명의 대출 신청 정보와 승인 여부(`Loan_Status`)입니다.

원본 데이터와 변수 설명은 [Kaggle Loan Prediction Problem Dataset](https://www.kaggle.com/datasets/altruistdelhite04/loan-prediction-problem-dataset) 페이지에서 확인하고 내려받을 수 있습니다. Kaggle에 로그인한 뒤 **Download**를 선택합니다. 압축을 푼 `train_u6lujuX_CVtuZ9i.csv`를 아래 코드에 적힌 경로에 두면 데이터 준비가 끝납니다.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../dataset/extracted/Loan Prediction Problem Dataset/train_u6lujuX_CVtuZ9i.csv')
df.shape


In [ ]:
df['Loan_Status'].value_counts(normalize=True)


**승인(Y) 68.7% : 거절(N) 31.3%**로 두 클래스의 비율이 다른 **불균형 데이터**입니다. Part 8~9에서는 이 비율을 다수 클래스 기준선으로 사용해 모델의 지표를 해석합니다.


## Part 3. 결측치 확인과 전처리 대상 지정

먼저 결측치의 위치와 개수를 확인합니다. 실제 대체값은 아직 계산하지 않습니다. 학습·검증 데이터를 나눈 뒤 Part 6의 Pipeline이 범주형 최빈값(mode)과 수치형 중앙값(median)을 **학습 데이터에서만** 계산합니다.


In [ ]:
df.isna().sum()


In [ ]:
categorical_features = [
    'Gender', 'Married', 'Dependents',
    'Education', 'Self_Employed', 'Property_Area',
]
numeric_features = [
    'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
    'Loan_Amount_Term', 'Credit_History',
]

df[categorical_features + numeric_features].isna().sum()


## Part 4. 원본 입력과 타겟 준비

`Loan_Status`(Y/N)를 모델이 이해할 수 있는 0/1 숫자로 바꿉니다. 입력 X는 아직 결측치가 있는 원본 형태로 둡니다. 범주형 인코딩은 데이터를 나눈 뒤 Pipeline 안에서 수행합니다.


In [ ]:
df['target'] = (df['Loan_Status'] == 'Y').astype(int)

X = df[categorical_features + numeric_features].copy()
y = df['target']
X.head()


## Part 5. 학습/검증 데이터 분리

`train_test_split`에 `stratify=y`를 넣습니다. 이 설정은 학습 데이터와 검증 데이터의 클래스 비율을 원본과 비슷하게 유지합니다. 설정하지 않으면 무작위 분할 과정에서 검증 데이터의 N(거절) 비율이 우연히 크게 달라질 수 있으므로, **불균형 데이터에서는 stratify를 사용하는 것이 안정적인 비교에 도움이 됩니다.**


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('학습:', X_train.shape, '검증:', X_val.shape)
print('학습 타겟 비율:', y_train.mean().round(3), '검증 타겟 비율:', y_val.mean().round(3))


## Part 6. Logistic Regression 학습

분류에서 가장 기본이 되는 모델입니다. 이름에 'Regression'이 들어가지만 회귀가 아니라 **분류** 모델입니다 — 각 클래스에 속할 확률을 계산해서, 0.5를 기준으로 승인/거절을 나눕니다.


In [ ]:
categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
preprocessor = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_features),
    ('numeric', numeric_pipeline, numeric_features),
])

model = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=5000)),
])
model.fit(X_train, y_train)
pred = model.predict(X_val)


## Part 7. 예측 결과와 Confusion Matrix

Part 1에서 손으로 계산했던 혼동행렬을 실제 예측 결과로 다시 작성합니다.


In [ ]:
cm = confusion_matrix(y_val, pred)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_xticklabels(['N(거절)', 'Y(승인)'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['N(거절)', 'Y(승인)'])
ax.set_xlabel('예측')
ax.set_ylabel('실제')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=14)
ax.set_title('Confusion Matrix')
plt.show()


## Part 8. accuracy / precision / recall / F1

Part 1과 같은 공식을 사용하며, 이번에는 sklearn 함수로 계산합니다.


In [ ]:
print('Accuracy :', accuracy_score(y_val, pred))
print('Precision:', precision_score(y_val, pred))
print('Recall   :', recall_score(y_val, pred))
print('F1       :', f1_score(y_val, pred))


In [ ]:
print(classification_report(y_val, pred, target_names=['N(거절)', 'Y(승인)']))


`classification_report`를 보면 **클래스별 지표가 서로 다르게** 나타납니다. Y(승인)의 recall은 거의 1에 가깝지만 N(거절)의 recall은 훨씬 낮습니다. 이는 실제 거절 건 중 상당수가 '승인'으로 예측되었다는 뜻입니다. accuracy와 함께 클래스별 지표를 살펴보면 이러한 오류 유형을 구체적으로 확인할 수 있습니다.


## Part 9. 불균형 데이터 확인 — 다수 클래스 기준선과 비교

Part 2에서 확인한 68.7% : 31.3%의 클래스 비율을 다시 사용합니다. **항상 다수 클래스(Y)만 예측하는 모델**과 비교하면, 학습한 모델이 기준선보다 지표를 얼마나 개선했는지 확인할 수 있습니다.


In [ ]:
dummy_pred = [1] * len(y_val)  # 항상 '승인'만 예측

print('다수 클래스 기준선 accuracy :', accuracy_score(y_val, dummy_pred))
print('다수 클래스 기준선 precision:', precision_score(y_val, dummy_pred))
print('다수 클래스 기준선 recall   :', recall_score(y_val, dummy_pred))
print()
print('실제 모델 accuracy   :', accuracy_score(y_val, pred))


다수 클래스만 예측하는 기준선도 비교적 높은 accuracy를 얻습니다. **따라서 불균형 데이터에서는 accuracy와 함께 다수 클래스 기준선 및 클래스별 precision과 recall을 확인하는 것이 좋습니다.** 이 비교를 통해 학습한 모델이 어떤 오류를 줄였고 어떤 한계가 남았는지 더 구체적으로 판단할 수 있습니다.


## 인사이트 정리 (예시)

- 타겟이 68.7% : 31.3%로 불균형합니다. `stratify` 없이 분할하면 검증 세트의 클래스 비율이 전체 데이터와 다르게 구성될 수 있습니다.
- accuracy는 86.2%이며, 다수 클래스 기준선의 accuracy는 69.1%입니다. 두 값을 비교하면 학습한 모델이 기준선에서 추가로 개선한 정도를 확인할 수 있습니다.
- N(거절) 클래스의 recall은 Y(승인)보다 뚜렷이 낮습니다. 모델이 다수 클래스인 Y 쪽으로 더 자주 예측하는 경향이 있음을 보여줍니다.
- 우선할 지표는 문제 상황에 따라 달라집니다. 실제 거절 대상자를 승인으로 분류하는 오류 비용이 더 크다면 N 클래스의 recall을 중요하게 살펴보는 것이 적절합니다.

과제(항공편 지연 예측)에서도 **결측치/샘플링 확인 → 인코딩 → 분류모델 학습 → 지표 계산 → 불균형 여부 언급** 순서를 참고해 분석을 이어갈 수 있습니다.
